In [24]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr


In [25]:
load_dotenv(override=True)


True

In [26]:
pushover_user=os.getenv('PUSHOVER_USER')
pushover_token=os.getenv('PUSHOVER_TOKEN')
pushover_url="https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with{pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover tokend found{pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts withu
Pushover tokend founda


In [27]:
def push(message):
    print(f"Push: {message}")
    payload={"user": pushover_user,"token":pushover_token,"message":message}
    requests.post(pushover_url,data=payload)

In [28]:
push("HEY!!")

Push: HEY!!


In [29]:
def record_user_details(email,name="Not provided",notes="not provided"):
    push(f"Recording intrest from{name} with email {email} and notes{notes}")
    return {"recorded":"ok"}

In [30]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded":"ok"}

In [31]:
record_user_details_json={
    "name":"record_user_details",
    "description":"Use this tool to record that a user is intresetd in beign touch and provided an email adress",
    "parameters":{
        "type":"object",
        "properties":{
            "email":{
                "type":"string",
                "description":"The email address of this user"
            },
            "name":{
                "type":"string",
                "description":"The user's name if they provided it"
            },
            "notes":{
                "type":"string",
                "descprition":"Any additional information about the converstation thats worth recording to give context"
            }

        },
        "required":["email"],
        "additional_properties":False
    }
}

In [32]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [33]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [34]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is intresetd in beign touch and provided an email adress',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name if they provided it"},
     'notes': {'type': 'string',
      'descprition': 'Any additional information about the converstation thats worth recording to give context'}},
    'required': ['email'],
    'additional_properties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question']

In [35]:



def handle_tool_calls(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name=tool_call.function.name
        arguments=json.loads(tool_call.function.arguments)
        print(f"Tool called:{tool_name}",flush=True)
        if tool_name=="record_user_details":
            result=record_user_details(**arguments)
        elif tool_name=="record_unknown_question":
            result=record_unknown_question(**arguments)
        
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    
    return results

In [36]:
globals()["record_unknown_question"]("This is a really hard question")

Push: Recording This is a really hard question asked that I couldn't answer


{'recorded': 'ok'}

In [37]:
def handle_tool_call(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name=tool_call.function.name
        arguments=json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}",flush=True)
        tool=globals().get(tool_name)
        result=tool(**arguments)if tool else{}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results

In [38]:
from os import name


reader=PdfReader("Linkedin.pdf")
linkedin=""
for page in reader.pages:
    text=page.extract_text()
    if text:
        linkedin+=text

with open("summary.txt","r",encoding="utf-8")as f:
    summary=f.read()

name="Vibha Kotian"

In [39]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [40]:
gemini_api_key=os.getenv('GEMINI_API_KEY')
open_router_key=os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GEMINI_BASE_URL="https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(base_url=GEMINI_BASE_URL,api_key=gemini_api_key)

In [41]:
def chat(message,history):
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    done=False
    while not done:
        
        response=gemini.chat.completions.create(model="gemini-2.5-flash",messages=messages,tools=tools)
        finish_reason=response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            message=response.choices[0].message
            tool_calls=message.tool_calls
            results=handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done=True
    
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called:record_unknown_question
Push: Recording what is your faviorate music asked that I couldn't answer
Tool called:record_user_details
Push: Recording intrest fromvibzee with email triptikotian@gmail.com and notesnot provided
